In [13]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

In [ ]:
#TN we're reading in from UrbanSIM
tnurbansim = pd.read_csv(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\1-raw\2050RTP-TDMFeedback_2050Block.csv")

In [6]:
col_dict = {'sector_0': 'EMP_OTHER', 'sector_1': 'EMP_SERVICE', 'sector_2': 'EMP_RETAIL', 'sector_3': 'EMP_OFFICE', 'geo_level_id': 'GEO_ID',
               'sector_4': 'EMP_MEDICAL', 'sector_5': 'EMP_INDUSTRIAL', 'sector_6': 'EMP_GOVERNMENT', 'sector_7': 
               'EMP_FOODSERVICE', 'sector_8': 'EMP_EDUCATION', 'household_population': 'HHPOP', 'sum_total_households': 'HH', 'sum_total_jobs': 'EMP'}
tnurbansim = tnurbansim.rename(columns = col_dict)
tnurbansim_filt = tnurbansim[['GEO_ID', 'HHPOP', 'EMP', 'EMP_OTHER', 'EMP_SERVICE', 'EMP_RETAIL', 'EMP_OFFICE', 
'EMP_MEDICAL', 'EMP_INDUSTRIAL', 'EMP_GOVERNMENT', 'EMP_FOODSERVICE', 'EMP_EDUCATION']]

In [8]:
#notably, these aren't rounded to whole jobs. Since we're hexing this I don't think it matters.
tnurbansim_filt.head(2)

,GEO_ID,HHPOP,EMP,EMP_OTHER,EMP_SERVICE,EMP_RETAIL,EMP_OFFICE,EMP_MEDICAL,EMP_INDUSTRIAL,EMP_GOVERNMENT,EMP_FOODSERVICE,EMP_EDUCATION
0,470831203003000,16.582708,2.0,1.0,0.0,0.0,0.0,0.036140,0.035221,0.498547,0.286435,0.030014
1,470831203003090,0.231629,2.0,1.0,0.0,0.0,0.0,0.058606,0.049506,0.517907,0.301769,0.029002


In [9]:
tnurbansim_filt['GEO_ID'] = tnurbansim_filt['GEO_ID'].astype(str)

In [17]:
#KY we will take the same allocations and do 2050 controls, so read in the base year with allocations as well as the WP smoothed totals
ky = gpd.read_file(r"G:\3-GitHub\FCLUCS_Scenario_Planning\2-outputs\2-geospatial\BaseYear.gpkg", layer="KY_HHPOP")
kycontrols = read_wp23_table('DemSmoothed')

In [32]:
kycontrols.head(2)

,NAME,Variable,2023 Census,2023 WP,2024,2025,2026,2027,2028,2029,...,2042,2043,2044,2045,2046,2047,2048,2049,2050,Last Update Pushed_20260812
0,"Cannon County, Tennessee",Population,15063,14854.0,15122.259259,15180.518519,15239.777778,15300.037037,15359.296296,15419.555556,...,16232.925926,16297.185185,16362.444444,16427.703704,16492.962963,16559.222222,16625.481481,16691.740741,16759.0,None
1,"Cheatham County, Tennessee",Population,42254,42068.0,42485.111111,42718.222222,42952.333333,43188.444444,43424.555556,43663.666667,...,46893.111111,47152.222222,47412.333333,47673.444444,47936.555556,48201.666667,48467.777778,48735.888889,49005.0,None


In [34]:
thelist = ['Trigg County, Kentucky', 'Christian County, Kentucky']
kyhorizon = kycontrols[['NAME', '2023 Census', '2050']].loc[kycontrols['NAME'].isin(thelist)]
kyhorizon['HHPOP_GROWTH'] = kyhorizon['2050'] - kyhorizon['2023 Census']

In [35]:
kyhorizon.head(2)

,NAME,2023 Census,2050,HHPOP_GROWTH
17,"Christian County, Kentucky",72032,70260.0,-1772.0
19,"Trigg County, Kentucky",14369,16279.0,1910.0


In [47]:
trigghorizoncontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Trigg County, Kentucky', '2050'].iloc[0])
christianhorizoncontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Christian County, Kentucky', '2050'].iloc[0])
triggcontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Trigg County, Kentucky', '2023 Census'].iloc[0])
christiancontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Christian County, Kentucky', '2023 Census'].iloc[0])
triggdiff = int(kyhorizon.loc[kyhorizon['NAME'] == 'Trigg County, Kentucky', 'HHPOP_GROWTH'].iloc[0])
christiandiff = int(kyhorizon.loc[kyhorizon['NAME'] == 'Christian County, Kentucky', 'HHPOP_GROWTH'].iloc[0])

In [48]:
controls = {'Trigg County, Kentucky': trigghorizoncontrol,'Christian County, Kentucky': christianhorizoncontrol}

ky['HHPOP_20501'] = ((ky['HHPOP_ALLOC'])/100) * ky['COUNTY'].map(controls)

In [49]:
ky.head(2)

,GEO_ID,COUNTY,COUNTYFP,HHPOP,HHPOP_ALLOC,geometry,HHPOP_2050,HHPOP_20501,HHPOP_20502
0,212219701003003,"Trigg County, Kentucky",21221,4,0.028594,"POLYGON ((-87.87697 36.96011, -87.87566 36.959...",4.654800,4.654800,0.546143
1,212219701003006,"Trigg County, Kentucky",21221,16,0.114376,"POLYGON ((-87.87628 36.95714, -87.87603 36.957...",18.619201,18.619201,2.184574


In [50]:
ky.loc[ky['COUNTY'] == 'Trigg County, Kentucky', 'HHPOP_20501'].sum()

np.float64(16279.0)

In [51]:
trigghorizoncontrol

16279

In [52]:
#method two would be taking the entire growth and giving the share of growth to the corresponding allocation... 
triggdiff

1910

In [53]:
christiandiff

-1772

In [54]:
controls = {'Trigg County, Kentucky': (triggdiff + triggcontrol),'Christian County, Kentucky': (christiandiff + christiancontrol)}

ky['HHPOP_20502'] = ((ky['HHPOP_ALLOC'])/100) * ky['COUNTY'].map(controls)

In [55]:
ky.head()

,GEO_ID,COUNTY,COUNTYFP,HHPOP,HHPOP_ALLOC,geometry,HHPOP_2050,HHPOP_20501,HHPOP_20502
0,212219701003003,"Trigg County, Kentucky",21221,4,0.028594,"POLYGON ((-87.87697 36.96011, -87.87566 36.959...",4.654800,4.654800,4.654800
1,212219701003006,"Trigg County, Kentucky",21221,16,0.114376,"POLYGON ((-87.87628 36.95714, -87.87603 36.957...",18.619201,18.619201,18.619201
2,212219701003009,"Trigg County, Kentucky",21221,12,0.085782,"POLYGON ((-87.93644 36.92322, -87.93629 36.923...",13.964401,13.964401,13.964401
3,212219701003013,"Trigg County, Kentucky",21221,11,0.078633,"POLYGON ((-87.8564 36.92651, -87.85621 36.9266...",12.800701,12.800701,12.800701
4,212219701003016,"Trigg County, Kentucky",21221,32,0.221603,"POLYGON ((-87.84762 36.92303, -87.84755 36.923...",36.074702,36.074702,36.074702


In [45]:
ky['HHPOP_20502'].sum()

np.float64(137.99999999999977)

In [57]:
ky['MethodDiff'] = ky['HHPOP_20501'] - ky['HHPOP_20502']

In [58]:
ky['MethodDiff'].describe()

count    3001.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: MethodDiff, dtype: float64

In [59]:
test = gpd.read_file(r"G:\DATA\REQUESTS_CLIENTS\GNRC\20250507_GNRC_JLUS\2-GIS\1-GNRC_CREATED\FCLUCS Scenario Planning Data\Continued Data Acquisition\Raw_Analysis_Data20260910.gdb")

c:\Users\jmccall\AppData\Local\miniforge3\envs\jlus\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured MultiLineString' is converted to 'MultiLineString'
  return ogr_read(
c:\Users\jmccall\AppData\Local\miniforge3\envs\jlus\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(
c:\Users\jmccall\AppData\Local\miniforge3\envs\jlus\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'Raw_Analysis_Data20260910.gdb': 'Ref_Study_Area_Composite_2025' (default), 'Ref_Study_Counties_2025', 'Ref_Municipal_Limits_2025', 'Fort_Campbell_Military_Installation', 'FCLUCS_ParcelFabric_UrbanFootprint', 'UBNFTPRNT_2025_GENLANDUSE_PARCEL_JH', 'Existing_Buildings_2026', 'Tennessee_Committed_Recent_Infrastructure_Projects_TACIR', 'GHSL_Population_2020_Blocks_1975_2030',